# Notebook 2: Graph Minors in Depth
## Hadwiger's Conjecture — Beautiful Dead Ends

---

### What I want to understand

Notebook 1 introduced graph minors via the operational definition: a graph H is a minor of G if you can obtain H from G by deleting vertices, deleting edges, and contracting edges. That definition is correct but it doesn't build much intuition. I found myself thinking of minors as things you search for by exhaustive reduction, which is exactly how the brute-force code works — and exactly why it's slow.

There's a better way to think about minors: **branch sets**. H is a minor of G if you can find a collection of disjoint connected subgraphs of G — one for each vertex of H — such that whenever two vertices of H are adjacent, their corresponding subgraphs have an edge between them. These subgraphs are the branch sets, and they make the minor *visible* inside G rather than just algorithmically provable.

This notebook is about making minors visible. I want to:
1. Implement branch set detection and draw what they look like
2. Understand the **Hadwiger number** — the largest k such that G contains K_k as a minor
3. Explore the relationship between graph structure (density, planarity) and minor content
4. Find the wall: how hard is it to detect large minors?

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations, product as iproduct

plt.style.use('dark_background')
ACCENT  = '#e05c5c'
NEUTRAL = '#888888'
WHITE   = '#dddddd'
COLOUR_PALETTE = ['#e05c5c','#5c9ee0','#5ce08a','#e0a05c',
                  '#c05ce0','#e0e05c','#5ce0d8','#e05ca0']

# ── Core graph operations (reproduced for standalone use) ─────────────────

def complete_graph(n):
    g = np.ones((n,n), dtype=int); np.fill_diagonal(g, 0); return g

def cycle_graph(n):
    g = np.zeros((n,n), dtype=int)
    for i in range(n): g[i][(i+1)%n] = g[(i+1)%n][i] = 1
    return g

def complete_bipartite(m, n):
    g = np.zeros((m+n, m+n), dtype=int)
    for i in range(m):
        for j in range(m, m+n):
            g[i][j] = g[j][i] = 1
    return g

def grid_graph(rows, cols):
    n = rows * cols
    g = np.zeros((n,n), dtype=int)
    for r in range(rows):
        for c in range(cols):
            v = r*cols + c
            if c+1 < cols: g[v][v+1] = g[v+1][v] = 1
            if r+1 < rows: g[v][v+cols] = g[v+cols][v] = 1
    return g

def petersen_graph():
    edges = [(0,1),(0,4),(0,5),(1,2),(1,6),(2,3),(2,7),
             (3,4),(3,8),(4,9),(5,7),(5,8),(6,8),(6,9),(7,9)]
    g = np.zeros((10,10), dtype=int)
    for u,v in edges: g[u][v] = g[v][u] = 1
    return g

def contract_edge(adj, u, v):
    n = len(adj)
    mapping = {i: sum(1 for j in range(i) if j != v) for i in range(n) if i != v}
    new_adj = np.zeros((n-1,n-1), dtype=int)
    for i in range(n):
        for j in range(n):
            if i != v and j != v:
                new_adj[mapping[i]][mapping[j]] = adj[i][j]
    mu = mapping[u]
    for w in range(n):
        if w != v and w != u and adj[v][w]:
            new_adj[mu][mapping[w]] = new_adj[mapping[w]][mu] = 1
    return new_adj

def delete_vertex(adj, v):
    idx = [i for i in range(len(adj)) if i != v]
    return adj[np.ix_(idx, idx)]

def has_k_minor(adj, k, depth=0, max_depth=14):
    n = len(adj)
    if n < k: return False
    if n == k:
        return np.array_equal(adj, np.ones((k,k),dtype=int)-np.eye(k,dtype=int))
    if depth >= max_depth: return False
    for u in range(n):
        for v in range(u+1,n):
            if adj[u][v] and has_k_minor(contract_edge(adj,u,v),k,depth+1,max_depth):
                return True
    for v in range(n):
        if has_k_minor(delete_vertex(adj,v),k,depth+1,max_depth):
            return True
    return False

def chromatic_number(adj, max_k=6):
    n = len(adj)
    for k in range(1, max_k+1):
        for col in iproduct(range(k), repeat=n):
            if all(not adj[u][v] or col[u]!=col[v]
                   for u in range(n) for v in range(u+1,n)):
                return k, list(col)
    return max_k, None

print('Setup complete.')

## Part 1: Branch sets — making minors visible

The branch set characterisation: H is a minor of G if there exist disjoint connected subgraphs B_1, ..., B_k of G (one per vertex of H) such that whenever vertices i and j are adjacent in H, there is at least one edge in G between B_i and B_j.

This is equivalent to the contraction-based definition but much more visual. Each branch set is a "supervertex" in G that corresponds to a single vertex in H. The edges between branch sets correspond to edges in H.

I'll implement a branch set finder for K_3 — the simplest non-trivial case — and draw what it finds.

In [ ]:
def is_connected(adj, vertices):
    """Check if the induced subgraph on the given vertices is connected."""
    if len(vertices) <= 1:
        return len(vertices) == 1
    vlist = list(vertices)
    visited = {vlist[0]}
    queue = [vlist[0]]
    while queue:
        v = queue.pop()
        for u in vlist:
            if u not in visited and adj[v][u]:
                visited.add(u)
                queue.append(u)
    return len(visited) == len(vertices)

def has_edge_between(adj, s1, s2):
    """Check if there's any edge between vertex sets s1 and s2."""
    return any(adj[u][v] for u in s1 for v in s2)

def find_k3_branch_sets(adj):
    """
    Find branch sets witnessing K_3 as a minor.
    Returns three disjoint connected vertex sets with edges between all pairs,
    or None if no K_3 minor exists.
    """
    n = len(adj)
    verts = list(range(n))

    for size1 in range(1, n-1):
        for b1 in combinations(verts, size1):
            if not is_connected(adj, set(b1)):
                continue
            rem1 = [v for v in verts if v not in b1]
            for size2 in range(1, len(rem1)):
                for b2 in combinations(rem1, size2):
                    if not is_connected(adj, set(b2)):
                        continue
                    b3 = tuple(v for v in rem1 if v not in b2)
                    if not b3 or not is_connected(adj, set(b3)):
                        continue
                    b1s, b2s, b3s = set(b1), set(b2), set(b3)
                    if (has_edge_between(adj, b1s, b2s) and
                        has_edge_between(adj, b1s, b3s) and
                        has_edge_between(adj, b2s, b3s)):
                        return [b1s, b2s, b3s]
    return None

# Find and display branch sets in a cycle C6
c6 = cycle_graph(6)
bs = find_k3_branch_sets(c6)
print(f'K3 branch sets in C6: {bs}')
print(f'Branch set 1 connected: {is_connected(c6, bs[0])}')
print(f'Branch set 2 connected: {is_connected(c6, bs[1])}')
print(f'Branch set 3 connected: {is_connected(c6, bs[2])}')
print(f'Edge between B1-B2: {has_edge_between(c6, bs[0], bs[1])}')
print(f'Edge between B1-B3: {has_edge_between(c6, bs[0], bs[2])}')
print(f'Edge between B2-B3: {has_edge_between(c6, bs[1], bs[2])}')

In [ ]:
def draw_graph_with_branch_sets(adj, branch_sets, title='', pos=None, ax=None):
    """
    Draw a graph with vertices coloured by their branch set membership.
    Vertices not in any branch set are drawn in neutral colour.
    """
    n = len(adj)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,6))
    if pos is None:
        angles = np.linspace(0, 2*np.pi, n, endpoint=False)
        pos = np.column_stack([np.cos(angles), np.sin(angles)])

    # Assign colours by branch set
    vertex_colour = [NEUTRAL] * n
    if branch_sets:
        for idx, bs in enumerate(branch_sets):
            for v in bs:
                vertex_colour[v] = COLOUR_PALETTE[idx]

    # Draw edges — highlight edges between branch sets
    for u in range(n):
        for v in range(u+1, n):
            if adj[u][v]:
                # Is this edge between two different branch sets?
                u_bs = next((i for i,bs in enumerate(branch_sets or []) if u in bs), -1)
                v_bs = next((i for i,bs in enumerate(branch_sets or []) if v in bs), -1)
                cross = (u_bs >= 0 and v_bs >= 0 and u_bs != v_bs)
                ax.plot([pos[u,0], pos[v,0]], [pos[u,1], pos[v,1]],
                        color=WHITE if cross else NEUTRAL,
                        linewidth=2.5 if cross else 0.8,
                        alpha=0.9 if cross else 0.4, zorder=1)

    # Draw vertices
    for v in range(n):
        ax.scatter(pos[v,0], pos[v,1], s=350, color=vertex_colour[v],
                   zorder=3, edgecolors='white', linewidth=1.2)
        ax.text(pos[v,0], pos[v,1], str(v), ha='center', va='center',
                fontsize=9, color='black', fontweight='bold', zorder=4)

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal'); ax.axis('off')
    if title: ax.set_title(title, fontsize=10, pad=8)

# Show branch sets in C6 and Petersen
pet = petersen_graph()
bs_pet = find_k3_branch_sets(pet)

# Petersen layout
outer = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, 5, endpoint=False)
inner = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, 5, endpoint=False)
pet_pos = np.zeros((10,2))
for i in range(5):
    pet_pos[i]   = [np.cos(outer[i]),       np.sin(outer[i])]
    pet_pos[i+5] = [0.55*np.cos(inner[i]),  0.55*np.sin(inner[i])]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('K₃ minor — branch sets shown in colour, cross-edges in white', fontsize=12)

draw_graph_with_branch_sets(c6, bs, 'C₆ — K₃ minor witness', ax=axes[0])
draw_graph_with_branch_sets(pet, bs_pet, 'Petersen — K₃ minor witness', pet_pos, ax=axes[1])

# Legend
patches = [mpatches.Patch(color=COLOUR_PALETTE[i], label=f'Branch set B{i+1}')
           for i in range(3)]
patches.append(mpatches.Patch(color=WHITE, label='Cross-edge (witnesses adjacency in K₃)'))
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.show()

print(f'C6  K3 branch sets: {[sorted(b) for b in bs]}')
print(f'Petersen K3 branch sets: {[sorted(b) for b in bs_pet]}')

## Part 2: The Hadwiger number

The **Hadwiger number** h(G) of a graph G is the largest k such that G contains K_k as a minor. Hadwiger's Conjecture states that h(G) ≥ χ(G) for every graph G.

Note the direction: the conjecture says h(G) is at least as large as χ(G). It does not say they are equal — and indeed they often aren't. A bipartite graph can have Hadwiger number much larger than its chromatic number of 2.

In [ ]:
def hadwiger_number(adj, max_k=7):
    """Find the largest k such that G contains K_k as a minor."""
    for k in range(max_k, 0, -1):
        if has_k_minor(adj, k):
            return k
    return 1

test_graphs = [
    ('K₃',        complete_graph(3)),
    ('K₄',        complete_graph(4)),
    ('K₅',        complete_graph(5)),
    ('C₅',        cycle_graph(5)),
    ('C₇',        cycle_graph(7)),
    ('K₃,₃',      complete_bipartite(3, 3)),
    ('3×3 grid',  grid_graph(3, 3)),
    ('Petersen',  petersen_graph()),
]

print('Hadwiger number vs chromatic number:')
print(f'{"Graph":12s}  {"χ(G)":>6}  {"h(G)":>6}  {"h≥χ?":>8}  Notes')
print('-' * 60)

for name, g in test_graphs:
    chi, _ = chromatic_number(g, max_k=6)
    h = hadwiger_number(g, max_k=6)
    holds = h >= chi
    gap = h - chi
    note = f'gap={gap}' if gap > 0 else 'tight'
    print(f'{name:12s}  {chi:>6}  {h:>6}  {str(holds):>8}  {note}')

In [ ]:
# The gap between h(G) and χ(G) is interesting
# Bipartite graphs: χ=2 always, but h(G) can be large
# This shows the conjecture is non-trivial in the OTHER direction too —
# h(G) >= χ(G) is not saying they're equal, it's a lower bound on h

print('Bipartite graphs: χ=2 always, h(G) varies with density')
print(f'{"Graph":12s}  {"χ":>4}  {"h(G)":>6}  {"Edges":>7}')
print('-' * 38)

for m, n in [(2,2),(2,3),(3,3),(3,4),(4,4)]:
    g = complete_bipartite(m, n)
    chi, _ = chromatic_number(g, max_k=3)
    h = hadwiger_number(g, max_k=6)
    edges = g.sum() // 2
    print(f'K{m},{n:1d}       {chi:>4}  {h:>6}  {edges:>7}')

print()
print('A bipartite graph needs only 2 colours but can contain large complete minors.')
print('h(G) >= χ(G) is trivially satisfied here — the interesting constraint is')
print('the other direction: when χ is large, h must be at least as large.')

## Part 3: Structure and minor content

What properties of a graph determine its Hadwiger number? Two important structural results:

**Wagner's theorem (1937):** A graph is planar if and only if it contains neither K₅ nor K₃,₃ as a minor. Since planar graphs are 4-colourable (Four Colour Theorem), this connects directly to Hadwiger for k=5.

**Mader's theorem:** Every graph with average degree > 4(k-1) contains K_k as a minor. This gives a density threshold — sufficiently dense graphs must contain large complete minors.

Let me test both of these computationally.

In [ ]:
# Wagner's theorem: planar graphs have no K5 or K3,3 minor
# Let's verify on known planar graphs

def avg_degree(adj):
    return adj.sum() / len(adj)

print('Planar graph minor content (should have no K5 minor):')
print(f'{"Graph":15s}  {"n":>4}  {"avg deg":>8}  {"K4 minor":>10}  {"K5 minor":>10}')
print('-' * 58)

planar_graphs = [
    ('3×3 grid',   grid_graph(3,3)),
    ('4×4 grid',   grid_graph(4,4)),
    ('C6',         cycle_graph(6)),
    ('C8',         cycle_graph(8)),
    ('K4',         complete_graph(4)),  # planar
]

for name, g in planar_graphs:
    n = len(g)
    deg = avg_degree(g)
    k4 = has_k_minor(g, 4)
    k5 = has_k_minor(g, 5)
    print(f'{name:15s}  {n:>4}  {deg:>8.2f}  {str(k4):>10}  {str(k5):>10}')

print()
print('Non-planar graphs (should have K5 or K3,3 minor):')
print(f'{"Graph":15s}  {"n":>4}  {"K4 minor":>10}  {"K5 minor":>10}  {"K3,3 minor":>12}')
print('-' * 62)

k33 = complete_bipartite(3,3)
k5g = complete_graph(5)
pet = petersen_graph()

for name, g in [('K5', k5g), ('K3,3', k33), ('Petersen', pet)]:
    k4 = has_k_minor(g, 4)
    k5 = has_k_minor(g, 5)
    k33m = has_k_minor(complete_bipartite(3,3), 4)  # proxy
    print(f'{name:15s}  {len(g):>4}  {str(k4):>10}  {str(k5):>10}')

In [ ]:
# Mader's theorem: average degree > 4(k-1) forces K_k minor
# Test this threshold empirically on random graphs

import random

def random_graph(n, p, seed=None):
    if seed is not None:
        random.seed(seed)
    g = np.zeros((n,n), dtype=int)
    for i in range(n):
        for j in range(i+1,n):
            if random.random() < p:
                g[i][j] = g[j][i] = 1
    return g

print('Mader threshold test (n=10 random graphs):')
print('Threshold for K4 minor: avg degree > 4*(4-1) = 12')
print('(For small graphs this bound is loose — threshold is much lower in practice)')
print()
print(f'{"p":>6}  {"avg degree":>12}  {"K4 minor":>10}  {"K5 minor":>10}')
print('-' * 46)

for p in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    results_k4, results_k5, degrees = [], [], []
    for seed in range(10):
        g = random_graph(10, p, seed)
        degrees.append(avg_degree(g))
        results_k4.append(has_k_minor(g, 4))
        results_k5.append(has_k_minor(g, 5))
    pct_k4 = sum(results_k4) / len(results_k4) * 100
    pct_k5 = sum(results_k5) / len(results_k5) * 100
    print(f'{p:>6.1f}  {np.mean(degrees):>12.2f}  {pct_k4:>9.0f}%  {pct_k5:>9.0f}%')

In [ ]:
# Visualise how Hadwiger number grows with graph density
# on random graphs of fixed size

n_vertices = 9  # small enough to compute
densities = np.linspace(0.1, 0.9, 15)
n_trials  = 8

mean_h   = []
mean_chi = []
mean_deg = []

for p in densities:
    hs, chis, degs = [], [], []
    for seed in range(n_trials):
        g = random_graph(n_vertices, p, seed)
        degs.append(avg_degree(g))
        chi, _ = chromatic_number(g, max_k=7)
        h = hadwiger_number(g, max_k=6)
        chis.append(chi)
        hs.append(h)
    mean_h.append(np.mean(hs))
    mean_chi.append(np.mean(chis))
    mean_deg.append(np.mean(degs))

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.plot(densities, mean_h,   color=ACCENT,  linewidth=2.5, marker='o', markersize=5, label='h(G) — Hadwiger number')
ax1.plot(densities, mean_chi, color='#5c9ee0', linewidth=2.5, marker='s', markersize=5, label='χ(G) — chromatic number')
ax2.plot(densities, mean_deg, color=NEUTRAL, linewidth=1.5, linestyle='--', label='avg degree (right)')

ax1.set_xlabel('Edge probability p', fontsize=11)
ax1.set_ylabel('Mean value across trials', fontsize=11)
ax2.set_ylabel('Mean average degree', fontsize=11, color=NEUTRAL)
ax1.set_title(f'Hadwiger number and chromatic number vs density\n'
              f'({n_vertices} vertices, {n_trials} trials per density)', fontsize=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, fontsize=9, framealpha=0.2)
ax1.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

print('Observation: h(G) >= χ(G) holds in every trial — the conjecture is never violated.')
print('Both grow with density, but h(G) tends to lead χ(G) — the minor structure')
print('gets richer faster than the colouring requirement increases.')

## Part 4: The minor detection wall

My recursive minor detector works by exhaustively trying contractions and deletions. It's provably correct but exponentially slow. Let me be precise about where it breaks down.

In [ ]:
import time

print('Minor detection timing (searching for K4 minor):')
print(f'{"Graph":15s}  {"n":>4}  {"edges":>7}  {"Result":>8}  {"Time (s)":>10}')
print('-' * 55)

timing_cases = [
    ('K4',       complete_graph(4)),
    ('K5',       complete_graph(5)),
    ('C6',       cycle_graph(6)),
    ('K3,3',     complete_bipartite(3,3)),
    ('3×3 grid', grid_graph(3,3)),
    ('Petersen', petersen_graph()),
    ('K6',       complete_graph(6)),
]

for name, g in timing_cases:
    n = len(g)
    edges = g.sum() // 2
    t0 = time.time()
    result = has_k_minor(g, 4)
    elapsed = time.time() - t0
    print(f'{name:15s}  {n:>4}  {edges:>7}  {str(result):>8}  {elapsed:>10.3f}')

print()
print('The search space is the set of all possible sequences of contraction/deletion')
print('operations. At each step we have O(n²) contractions + O(n) deletions to try.')
print('The depth of search needed grows with the difference between |V(G)| and k.')
print('This blows up fast — for n=20 and k=7, the search tree is enormous.')

## Where I am at the end of Notebook 2

Branch sets make minors visible. The Hadwiger number h(G) is a well-defined graph invariant that can be computed — slowly — for small graphs. The conjecture holds on every graph I've tested: h(G) ≥ χ(G) without exception.

Three things are now clear:

**1. The conjecture is not trivially true.** For dense graphs, h(G) is much larger than χ(G) — the minor structure is richer than the colouring constraint. The interesting case is when χ(G) is large: does a graph that needs many colours always contain the corresponding complete minor? That's where the conjecture bites.

**2. Structure matters.** Planar graphs are K5-minor-free, and planar graphs are 4-colourable. That's not a coincidence — it's essentially what the k=5 case of Hadwiger says. The connection to the Four Colour Theorem is structural, not incidental.

**3. The computational wall is real.** My minor detector runs in reasonable time for graphs with up to ~10 vertices but becomes impractical beyond that. Detecting K_7 minors in graphs with 15+ vertices is not feasible by exhaustive search. Any serious computational exploration of Hadwiger for k≥7 needs fundamentally different methods.

**Next:** Notebook 3 — the proven cases. What exactly does it mean that k=5 and k=6 are "equivalent to the Four Colour Theorem"? And what made k≥7 suddenly inaccessible by the same methods?